### Import data

In [1]:
import sys
import pandas as pd
from pathlib import Path

dossier_pipeline = Path.cwd() / "lab" / "pipeline_template"

sys.path.insert(0, str(dossier_pipeline))

from chemins_projet import CSV_BRUT

# racine_lab = Path(__file__).resolve().parents[1]
# dossier_exemples = racine_lab / "outputs" / "exemples_snippets"
# dossier_exemples.mkdir(parents=True, exist_ok=True)

donnees_brutes = pd.read_csv(
    CSV_BRUT,
    dtype="string",
    keep_default_na=False,
)
print(donnees_brutes)

     ticket_id channel resolution_minutes
0     TKT-1450   email               36.6
1     TKT-1691   email               27.2
2     TKT-0268   phone               82.0
3     TKT-0273   phone              114.2
4     TKT-0921    chat              209.0
...        ...     ...                ...
1795  TKT-1657   email               29.3
1796  TKT-0647    chat               53.3
1797  TKT-0492   phone               59.6
1798  TKT-1444   email               28.3
1799  TKT-0622    chat               78.7

[1800 rows x 3 columns]


### Structure

In [2]:
print("Dimensions :", donnees_brutes.shape)
print("-"*50)
# print("Structure :", donnees_brutes.describe)
# print("-"*50)
print("Types chargés :")
print(donnees_brutes.dtypes)
print("-"*50)

print("Différents canaux :")
print(donnees_brutes["channel"].value_counts(dropna=False))
print("-"*50)

for colonne in donnees_brutes.columns:
    valeurs = donnees_brutes[colonne]
    cellules_vides = valeurs.str.strip() == ""
    # possible parce dtype=string à l'import
    nombre_vides = int(cellules_vides.sum())
    print(colonne, ":", nombre_vides, "cellule(s) vide(s)")

print("-"*50)
identifiants_repetes = donnees_brutes["ticket_id"].duplicated(keep=False)
print("Lignes portant un identifiant répété :")
doublons = donnees_brutes.loc[identifiants_repetes]
print(doublons)
print(doublons.nunique(axis=1))

Dimensions : (1800, 3)
--------------------------------------------------
Types chargés :
ticket_id             string
channel               string
resolution_minutes    string
dtype: object
--------------------------------------------------
Différents canaux :
channel
phone      600
chat       600
email      599
 Email       1
Name: count, dtype: Int64
--------------------------------------------------
ticket_id : 0 cellule(s) vide(s)
channel : 0 cellule(s) vide(s)
resolution_minutes : 0 cellule(s) vide(s)
--------------------------------------------------
Lignes portant un identifiant répété :
     ticket_id channel resolution_minutes
131   TKT-0099   phone               93.1
1444  TKT-0099   phone               93.1
131     3
1444    3
dtype: int64


#### Copie + ligne source

In [3]:
travail = donnees_brutes.copy()
travail["ligne_source"] = range(2, len(travail) + 2)
# pour faire correspondre les première lignes de chaques format :
# ligne 0 du df -> la 2 du csv (en-tête étant la 1)

#### Valeur(s) négative(s)

In [4]:
texte_duree = travail["resolution_minutes"].str.strip()
duree_numerique = pd.to_numeric(texte_duree, errors="coerce")
# coerce = permet de transformé les valeurs non compatibles en valeurs manquantes
travail["resolution_minutes"] = duree_numerique

cellule_vide = texte_duree == ""
conversion_impossible = (texte_duree != "") & duree_numerique.isna()
duree_negative = duree_numerique < 0

print("Durées absentes :")
display(travail.loc[cellule_vide])
print("-"*50)

print("Textes non convertibles :")
display(donnees_brutes.loc[conversion_impossible])
print("-"*50)

print("Durées négatives :")
display(travail.loc[duree_negative])

Durées absentes :


,ticket_id,channel,resolution_minutes,ligne_source


--------------------------------------------------
Textes non convertibles :


,ticket_id,channel,resolution_minutes


--------------------------------------------------
Durées négatives :


,ticket_id,channel,resolution_minutes,ligne_source
918,TKT-0668,chat,-25.0,920


### Nettoyage data

#### Création journal

In [5]:
from chemins_projet import CSV_JOURNAL_QUALITE

journal = []

colonnes_journal = [
    "ligne_source", "ticket_id", "regle", "decision", "justification"
]

#### Valeur négative

In [6]:
masque_duree_invalide = cellule_vide | conversion_impossible | duree_negative

for indice in travail.index[masque_duree_invalide]:
    if cellule_vide.loc[indice]:
        regle = "duree_absente"
        decision = "exclue"
        justification = "La durée est vide."

    elif conversion_impossible.loc[indice]:
        regle = "duree_non_numerique"
        decision = "exclue"
        justification = "La durée ne peut pas être convertie en nombre."

    else:
        ancienne_valeur = travail.loc[indice, "resolution_minutes"]
        nouvelle_valeur = abs(ancienne_valeur)

        travail.loc[indice, "resolution_minutes"] = nouvelle_valeur
        
        regle = "duree_negative"
        decision = "corrigee"
        justification = (
            f"Valeur négative interprétée comme une erreur de saisie : "
            f"{ancienne_valeur} devient {abs(nouvelle_valeur)}."
        )

    journal.append({
        "ligne_source": int(travail.loc[indice, "ligne_source"]),
        "ticket_id": travail.loc[indice, "ticket_id"],
        "regle": regle,
        "decision": decision,
        "justification": justification,
    })

display(pd.DataFrame(journal))

,ligne_source,ticket_id,regle,decision,justification
0,920,TKT-0668,duree_negative,corrigee,Valeur négative interprétée comme une erreur d...


#### Capitale dans les canaux

In [7]:
canal_avant = travail["channel"]
canal_apres = canal_avant.str.strip().str.casefold()

masque_canal_corrige = canal_avant != canal_apres

for indice in travail.index[masque_canal_corrige]:
    ancienne_valeur = travail.loc[indice, "channel"]
    nouvelle_valeur = canal_apres.loc[indice]

    travail.loc[indice, "channel"] = nouvelle_valeur

    journal.append({
        "ligne_source": int(travail.loc[indice, "ligne_source"]),
        "ticket_id": travail.loc[indice, "ticket_id"],
        "regle": "uniformisation casse",
        "decision": "corrigee",
        "justification": (
            f"Le canal « {ancienne_valeur} » est normalisée en "
            f"« {nouvelle_valeur} »."
        ),
    })

display(pd.DataFrame(journal))

,ligne_source,ticket_id,regle,decision,justification
0,920,TKT-0668,duree_negative,corrigee,Valeur négative interprétée comme une erreur d...
1,538,TKT-1238,uniformisation casse,corrigee,Le canal « Email » est normalisée en « email ».


#### Uniformisation ID

In [8]:
schema_ticket_id = r"TKT-[0-9]{4}"

masque_ticket_invalide = ~travail["ticket_id"].str.fullmatch(
    schema_ticket_id,
    na=False,
)

display(
    travail.loc[
        masque_ticket_invalide,
        ["ligne_source", "ticket_id"]
    ]
)

for indice in travail.index[masque_ticket_invalide]:
    valeur = travail.loc[indice, "ticket_id"]

    journal.append({
        "ligne_source": int(travail.loc[indice, "ligne_source"]),
        "ticket_id": valeur,
        "regle": "format_ticket_id",
        "decision": "exclue",
        "justification": (
            "L'identifiant ne respecte pas le format TKT- suivi de "
            "quatre chiffres."
        ),
    })

display(pd.DataFrame(journal))

,ligne_source,ticket_id


,ligne_source,ticket_id,regle,decision,justification
0,920,TKT-0668,duree_negative,corrigee,Valeur négative interprétée comme une erreur d...
1,538,TKT-1238,uniformisation casse,corrigee,Le canal « Email » est normalisée en « email ».


#### Suppression de la ligne en doublon

In [9]:
# True pour toutes les lignes appartenant à un doublon
masque_doublons = travail["ticket_id"].duplicated(keep=False)
print("Les deux lignes dupliquées")
display(travail.loc[masque_doublons])

print("-" * 50 + "\n")

# True uniquement pour les doublons après le premier
masque_a_supprimer = travail["ticket_id"].duplicated(keep="first")
print("La ligne a supprimer : la deuxième")
display(travail.loc[masque_a_supprimer])
print("-" * 50 + "\n")

for indice in travail.index[masque_a_supprimer]:
    decision = {
        "ligne_source": int(travail.loc[indice, "ligne_source"]),
        "ticket_id": travail.loc[indice, "ticket_id"],
        "regle": "identifiant_duplique",
        "decision": "exclue",
        "justification": (
            "Deuxième occurrence du ticket ; première occurrence conservée"
        ),
    }
    journal.append(decision)

travail = travail.loc[~masque_a_supprimer]

display(pd.DataFrame(journal))


Les deux lignes dupliquées


,ticket_id,channel,resolution_minutes,ligne_source
131,TKT-0099,phone,93.1,133
1444,TKT-0099,phone,93.1,1446


--------------------------------------------------

La ligne a supprimer : la deuxième


,ticket_id,channel,resolution_minutes,ligne_source
1444,TKT-0099,phone,93.1,1446


--------------------------------------------------



,ligne_source,ticket_id,regle,decision,justification
0,920,TKT-0668,duree_negative,corrigee,Valeur négative interprétée comme une erreur d...
1,538,TKT-1238,uniformisation casse,corrigee,Le canal « Email » est normalisée en « email ».
2,1446,TKT-0099,identifiant_duplique,exclue,Deuxième occurrence du ticket ; première occur...


### Export journal

In [10]:
journal_df = pd.DataFrame(journal, columns=colonnes_journal)
journal_df.to_csv( CSV_JOURNAL_QUALITE, index=False)
display(journal_df)

,ligne_source,ticket_id,regle,decision,justification
0,920,TKT-0668,duree_negative,corrigee,Valeur négative interprétée comme une erreur d...
1,538,TKT-1238,uniformisation casse,corrigee,Le canal « Email » est normalisée en « email ».
2,1446,TKT-0099,identifiant_duplique,exclue,Deuxième occurrence du ticket ; première occur...


### Description

In [12]:
display(travail.head(5))

,ticket_id,channel,resolution_minutes,ligne_source
0,TKT-1450,email,36.6,2
1,TKT-1691,email,27.2,3
2,TKT-0268,phone,82.0,4
3,TKT-0273,phone,114.2,5
4,TKT-0921,chat,209.0,6


In [25]:
def resumer_canaux(donnees):
    lignes_resume = []
    canaux = donnees["channel"].unique()

    for canal in canaux:
        appartient_au_cannal = donnees["channel"] == canal
        tickets_du_canal = donnees.loc[appartient_au_cannal]
        durees = tickets_du_canal["resolution_minutes"]

        ligne = {
            "channel": canal,
            "nombre": len(durees),
            "moyenne": durees.mean(),
            "mediane": durees.median(),
            "ecart_type": durees.std(ddof=1),
            "q1": durees.quantile(0.25),
            "q3": durees.quantile(0.75),
            "minimum": durees.min(),
            "maximum": durees.max(),
        }

        lignes_resume.append(ligne)

    return pd.DataFrame(lignes_resume)

resume_canaux = resumer_canaux(travail)
display(resume_canaux)

,channel,nombre,moyenne,mediane,ecart_type,q1,q3,minimum,maximum
0,email,600,91.823167,70.55,63.629380,44.075,122.925,25.1,418.0
1,phone,599,89.671452,89.30,18.772966,76.850,101.750,9.1,158.0
2,chat,600,89.799833,82.30,42.678326,60.275,108.750,24.8,320.1


In [ ]:
    histogramme = px.histogram(
        tickets_nettoyes,
        x="resolution_minutes",
        color="channel",
        facet_row="channel",
        facet_row_spacing=0.08,
        marginal="rug",
        # text_auto=True,
        template="plotly_white",
        color_discrete_sequence=["#008C95", "#F06B52", "#5B6CFF"],
        title="Durées de résolution par canal",
        labels={
            "resolution_minutes": "Durée de résolution (minutes)",
            "channel": "Canal",
        },
        range_x=[0, 430],
    )

    histogramme.update_traces(
        xbins=dict(
            start=5,
            end=425,
            size=10,
        ),
        marker_line_width=1,
        marker_line_color="white",
        selector=dict(type="histogram"),
    )

    histogramme.update_traces(
        marker_size=10,
        marker_line_width=1,
        selector=dict(type="box"),
    )

    histogramme.update_layout(
        height=1100,
        font_size=14,
        showlegend=False,
        margin=dict(
            t=120,
            b=80,
            l=80,
            r=40,
        ),
    )

    histogramme.update_xaxes(
        title_text="Durée de résolution (minutes)",
        range=[0, 430],
    )

    histogramme.update_yaxes(
        title_text="Nombre de tickets",
    )

    histogramme.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.replace("Canal=", "")
    )
)
